In [2]:
!pip install openai

In [3]:
import re
import numpy as np
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Initialize the official OpenAI client
client = OpenAI(api_key="")

def split_into_sentences(text: str) -> list[str]:
    """Splits a body of text into individual clean sentences."""
    # Split text at sentence boundary punctuation
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Generates embeddings using OpenAI's modern API."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [data.embedding for data in response.data]

def semantic_chunking(text: str, percentile_threshold: float = 85.0) -> list[str]:
    """Chunks text semantically by evaluating vector distance shifts."""
    sentences = split_into_sentences(text)
    if len(sentences) < 2:
        return sentences

    # Step 1: Vectorize sentences
    embeddings = get_embeddings(sentences)

    # Step 2: Compute cosine distances between consecutive sentences
    distances = []
    for i in range(len(embeddings) - 1):
        vec1 = np.array(embeddings[i]).reshape(1, -1)
        vec2 = np.array(embeddings[i+1]).reshape(1, -1)

        # Calculate similarity and convert to a distance metric
        similarity = cosine_similarity(vec1, vec2)[0][0]
        distances.append(1 - similarity)

    # Step 3: Define a dynamic breakpoint threshold based on percentiles
    breakpoint_distance_threshold = np.percentile(distances, percentile_threshold)

    # Step 4: Build chunks where distance shifts exceed our target threshold
    chunks = []
    current_chunk = [sentences[0]]

    for i, distance in enumerate(distances):
        if distance > breakpoint_distance_threshold:
            # High semantic distance means a topic shift occurred
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i + 1]]
        else:
            # Low semantic distance means topic is continuing
            current_chunk.append(sentences[i + 1])

    chunks.append(" ".join(current_chunk))
    return chunks


In [8]:
# --- Example Usage ---
document_text = (
    "The planet Mars has fascinated humans for centuries. Scientists actively study its "
    "dry, rocky surface and atmosphere. NASA has sent multiple robotic rovers to explore "
    "its red terrain. Shifting to corporate finances, Apple reported record-breaking "
    "earnings this quarter. Their stock surged due to high demands for consumer hardware. "
    "Investors are closely watching tech market volatility."
)

In [9]:

final_chunks = semantic_chunking(document_text, percentile_threshold=80)
for idx, chunk in enumerate(final_chunks):
    print(f"--- Chunk {idx + 1} ---")
    print(chunk)

--- Chunk 1 ---
The planet Mars has fascinated humans for centuries. Scientists actively study its dry, rocky surface and atmosphere. NASA has sent multiple robotic rovers to explore its red terrain.
--- Chunk 2 ---
Shifting to corporate finances, Apple reported record-breaking earnings this quarter. Their stock surged due to high demands for consumer hardware. Investors are closely watching tech market volatility.
